# 09 — Programmatically Use the Genie Portfolio Agent

In Notebook 08, we created the governed portfolio semantic layer:

```text
investment_ai.portfolio.portfolio_metrics
```

We then created and curated the **Investment AI Portfolio Agent** in the Genie UI.

In this notebook, we connect to that Genie Agent from Python and turn it into a reusable structured-data capability for our larger multi-agent system.

### Architecture

```text
Natural-language portfolio question
              ↓
Python / future Multi-Agent Orchestrator
              ↓
Investment AI Portfolio Agent
              ↓
portfolio_metrics
              ↓
Governed SQL + SQL Warehouse
              ↓
Structured portfolio answer
```

The key idea is that Python does **not** generate portfolio SQL itself.

The curated Genie Agent owns the structured analytics experience, and our code simply calls it.

## 1. Install the Databricks AI Bridge client

Databricks recommends AI Bridge when Python needs to query one specific curated Genie Agent.

AI Bridge handles the underlying Genie API calls, polling, and result retrieval for us.

In [0]:
%pip install -q --upgrade databricks-ai-bridge
dbutils.library.restartPython()

## 2. Configure stable MLflow tracking

AI Bridge emits MLflow traces for Genie calls.

We use an explicit workspace experiment so the notebook does not depend on a temporary notebook-associated experiment.

In [0]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/investment_ai_portfolio_agent")

print("✅ MLflow tracking configured.")
print("Experiment: /Shared/investment_ai_portfolio_agent")

## 3. Configure the Genie Portfolio Agent

The current Genie UI calls this value an **Agent ID**.

The AI Bridge Python client still uses the parameter name `space_id` for compatibility with the earlier Genie Space API naming.

In [0]:
from databricks_ai_bridge.genie import Genie

GENIE_AGENT_ID = "01f1cb3f9bdd1714ae1bb5ad29f10f32"
GENIE_AGENT_NAME = "Investment AI Portfolio Agent"

genie = Genie(
    space_id=GENIE_AGENT_ID,
    return_pandas=False,
)

print("✅ Genie client created.")
print("Agent :", GENIE_AGENT_NAME)
print("ID    :", GENIE_AGENT_ID)

## 4. Create a small response helper

A Genie response can contain more than just the final answer.

For development and debugging, it is useful to see:

- the business answer,
- the SQL Genie generated,
- the conversation ID,
- any suggested follow-up questions.

This helps us understand what the structured-data agent actually did.

In [0]:
def display_genie_response(response):
    print("=" * 80)
    print("GENIE ANSWER")
    print("=" * 80)
    print(response.result)

    query = getattr(response, "query", None)
    if query:
        print("\n" + "=" * 80)
        print("GENERATED SQL")
        print("=" * 80)
        print(query)

    conversation_id = getattr(response, "conversation_id", None)
    if conversation_id:
        print("\nConversation ID:", conversation_id)

    suggested = getattr(response, "suggested_questions", None)
    if suggested:
        print("\nSuggested follow-up questions:")
        for question in suggested:
            print("-", question)

## 5. Test the first portfolio question

Now we call the same Genie Agent we configured in the UI.

The question is written in normal business language.

Genie is responsible for interpreting that question using its instructions, curated examples, and the governed Metric View.

In [0]:
question = (
    "How much do we own of Synthetic Compute Company 01 "
    "as of June 30, 2026?"
)

ownership_response = genie.ask_question(question)

display_genie_response(ownership_response)

## 6. What should happen

For this question, Genie should use the governed portfolio semantic layer to determine:

- the relevant company/security,
- the requested portfolio snapshot date,
- total quantity or shares owned,
- total market value.

The generated SQL should query the curated `portfolio_metrics` Metric View rather than bypassing the semantic layer and querying raw portfolio tables directly.

## 7. Continue the same conversation

Genie supports stateful conversations.

Instead of starting from zero for every question, we can keep the `conversation_id` and ask a follow-up question using the context from the previous turn.

This becomes useful later when our application supports a real conversational experience.

In [0]:
conversation_id = ownership_response.conversation_id

follow_up_question = (
    "Break that ownership down by portfolio."
)

follow_up_response = genie.ask_question(
    follow_up_question,
    conversation_id=conversation_id,
)

display_genie_response(follow_up_response)

## 8. Create the reusable Portfolio Agent interface

The larger multi-agent system should not need to know the details of the Genie API.

We therefore create one small Python function that represents the structured Portfolio Agent capability.

Later, the Multi-Agent Orchestrator can call this function as a tool.

In [0]:
def ask_portfolio_agent(
    question: str,
    conversation_id: str | None = None,
) -> dict:
    """Ask the curated Investment AI Portfolio Agent a structured-data question."""

    response = genie.ask_question(
        question,
        conversation_id=conversation_id,
    )

    return {
        "answer": response.result,
        "sql": getattr(response, "query", None),
        "conversation_id": response.conversation_id,
        "suggested_questions": getattr(
            response,
            "suggested_questions",
            None,
        ),
    }

print("✅ Reusable Portfolio Agent interface created.")

## 9. Test the reusable interface

This is the interface we care about from an architecture perspective.

The caller supplies a business question.

The Portfolio Agent returns a structured result that our future orchestrator can consume.

In [0]:
portfolio_result = ask_portfolio_agent(
    "What is our market value exposure by sector as of June 30, 2026?"
)

print("Answer:")
print(portfolio_result["answer"])

print("\nGenerated SQL:")
print(portfolio_result["sql"])

print("\nConversation ID:")
print(portfolio_result["conversation_id"])

## 10. Test the scope boundary

The Portfolio Agent is intentionally specialized.

It should answer structured questions about holdings, ownership, valuation, and exposure.

It should **not** become our research analyst.

This test checks whether the instructions we configured in the Genie UI are actually enforcing that boundary.

In [0]:
boundary_question = (
    "Why did Synthetic Compute Company 01 "
    "lower its outlook in Q2 2026?"
)

boundary_response = genie.ask_question(boundary_question)

display_genie_response(boundary_response)

## 11. Why the boundary matters

At this point, our POC has two clearly separated capabilities:

```text
Research Agent
    ↓
Unstructured research evidence
    ↓
Why did something happen?


Portfolio Agent
    ↓
Structured governed analytics
    ↓
How much do we own?
What is it worth?
What is our exposure?
```

A user can eventually ask a combined question such as:

> Which companies lowered their outlook in Q2 2026, and how much do we own?

The future Multi-Agent Orchestrator will split that request across the two specialized capabilities and combine the results.

# ✅ Notebook 09 complete

We now have a programmatic structured-data capability:

```text
Python
   ↓
ask_portfolio_agent()
   ↓
Databricks AI Bridge
   ↓
Investment AI Portfolio Agent
   ↓
portfolio_metrics
   ↓
Governed SQL
   ↓
Portfolio answer
```

### What we accomplished

- Connected Python to the curated Genie Portfolio Agent.
- Used AI Bridge rather than manually implementing Genie API polling.
- Tested a natural-language ownership question.
- Demonstrated stateful follow-up conversation.
- Wrapped Genie behind a reusable `ask_portfolio_agent()` interface.
- Tested structured sector analytics.
- Tested the Research-vs-Portfolio scope boundary.
- Preserved a fresh question for the live YouTube demo.

### Next

**Notebook 10 — Build the Multi-Agent Orchestrator**

That notebook will connect:

```text
Research Agent
        +
Portfolio Agent
        ↓
Multi-Agent Orchestrator
        ↓
One coordinated answer
```

The orchestrator will decide which specialized capability to call and how to combine their results.